# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nandhanamj/flyrank_ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I will prioritize content items for refresh using two observed signals:

1. **Staleness** — older content gets a higher refresh score because it has had more time since its last observed update.
2. **Search visibility** — content with weaker observed search visibility gets a higher refresh score.

The rule adds points for higher staleness and lower search visibility. Items with higher scores are ranked earlier in the refresh queue.

**Reason codes:**
- `STALE` — staleness contributed to the score.
- `LOW_VISIBILITY` — low search visibility contributed to the score.
- `STALE_LOW_VISIBILITY` — both signals contributed.
- `REVIEW` — neither signal was strong enough for a specific reason code.

**Action labels:**
- `REFRESH` — prioritize the item for refresh review.
- `REVIEW` — keep the item in the queue for human review.

In [20]:
%pip -q install duckdb

In [21]:
import duckdb

con = duckdb.connect()

print("DuckDB connection ready.")

DuckDB connection ready.


In [22]:
#HF authentication:

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face authentication ready.")

Hugging Face authentication ready.


In [23]:
PERF_MARCH = (
    "read_parquet("
    "'hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet'"
    ")"
)

print("March 2026 performance partition ready.")

March 2026 performance partition ready.


In [24]:
# Inspect the content dimension schema for a real staleness/update signal

DIM_CONTENT = (
    "read_parquet("
    "'hf://datasets/FlyRank/internship-warehouse/"
    "dim_content.parquet'"
    ")"
)

print(con.execute(f"""
    DESCRIBE SELECT *
    FROM {DIM_CONTENT}
    LIMIT 1
""").df().to_string(index=False))

               column_name column_type null  key default extra
            client_hash_id     VARCHAR  YES None    None  None
           content_hash_id     VARCHAR  YES None    None  None
           keyword_hash_id     VARCHAR  YES None    None  None
               url_hash_id     VARCHAR  YES None    None  None
        keyword_char_count      BIGINT  YES None    None  None
       keyword_token_count      BIGINT  YES None    None  None
            url_char_count      BIGINT  YES None    None  None
      content_created_date        DATE  YES None    None  None
      content_updated_date        DATE  YES None    None  None
              content_type     VARCHAR  YES None    None  None
             search_volume      BIGINT  YES None    None  None
               competition      DOUBLE  YES None    None  None
         competition_level     VARCHAR  YES None    None  None
                       cpc      DOUBLE  YES None    None  None
               main_intent     VARCHAR  YES None    Non

### Signal 1 — Content staleness

I will check whether older content is associated with lower March search performance.
I use `content_updated_date` to measure how long it has been since a content item was updated.

This is linked to FlyRank's refresh flags because staleness is one of the signals behind refresh prioritization.

**Verdict: CONFIRMED**

Observed March performance was lower in every older staleness bucket:
- <90 days: 2.53 average clicks (n=324,013)
- 90–179 days: 0.36 average clicks (n=3,608)
- 180–364 days: 0.01 average clicks (n=3,816)

This supports using content staleness as a directional refresh-prioritization signal. The oldest bucket measured here had very low observed March clicks, although the bucket sizes are uneven.

In [25]:
# Signal 1: Content staleness vs March search performance

staleness_buckets = con.execute(f"""
WITH content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        DATE_DIFF(
            'day',
            content_updated_date,
            DATE '2026-03-31'
        ) AS days_stale
    FROM {DIM_CONTENT}
    WHERE content_updated_date IS NOT NULL
),
performance AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS march_clicks
    FROM {PERF_MARCH}
    GROUP BY 1, 2
),
bucketed AS (
    SELECT
        c.*,
        p.march_clicks,
        CASE
            WHEN days_stale < 90 THEN '<90 days'
            WHEN days_stale < 180 THEN '90-179 days'
            WHEN days_stale < 365 THEN '180-364 days'
            ELSE '365+ days'
        END AS staleness_bucket
    FROM content c
    JOIN performance p
      USING (client_hash_id, content_hash_id)
)
SELECT
    staleness_bucket,
    COUNT(*) AS n,
    ROUND(AVG(march_clicks), 2) AS avg_march_clicks
FROM bucketed
GROUP BY staleness_bucket
ORDER BY
    CASE staleness_bucket
        WHEN '<90 days' THEN 1
        WHEN '90-179 days' THEN 2
        WHEN '180-364 days' THEN 3
        ELSE 4
    END
""").df()

staleness_buckets

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,avg_march_clicks
0,<90 days,324013,2.53
1,90-179 days,3608,0.36
2,180-364 days,3816,0.01


### Signal 2 — Search visibility

I will check whether content with lower March GSC impressions also has lower March GSC clicks.
This is an observed search-performance signal available at the decision point.

**Verdict: CONFIRMED**

Observed March clicks increased across every search-visibility bucket:
- 0 impressions: 0.00 average clicks (n=154,699)
- 1–99 impressions: 0.09 average clicks (n=75,297)
- 100–999 impressions: 0.94 average clicks (n=56,383)
- 1000+ impressions: 16.92 average clicks (n=45,058)

This supports using March search visibility as a directional prioritization signal. The relationship is measured within the March observation window and is not a future outcome.

In [26]:
# Signal 2: Search visibility vs March search performance

visibility_buckets = con.execute(f"""
WITH performance AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks
    FROM {PERF_MARCH}
    GROUP BY 1, 2
)
SELECT
    CASE
        WHEN march_impressions = 0 THEN '0 impressions'
        WHEN march_impressions < 100 THEN '1-99'
        WHEN march_impressions < 1000 THEN '100-999'
        ELSE '1000+'
    END AS visibility_bucket,
    COUNT(*) AS n,
    ROUND(AVG(march_clicks), 2) AS avg_march_clicks
FROM performance
GROUP BY 1
ORDER BY
    CASE visibility_bucket
        WHEN '0 impressions' THEN 1
        WHEN '1-99' THEN 2
        WHEN '100-999' THEN 3
        ELSE 4
    END
""").df()

visibility_buckets

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,visibility_bucket,n,avg_march_clicks
0,0 impressions,154699,0.00
1,1-99,75297,0.09
2,100-999,56383,0.94
3,1000+,45058,16.92


## 2. Build the ranked queue (writes the CSV)

### Baseline rule

I score each content item using two observed signals at the March 31, 2026 decision point:

- **Staleness:** older content receives more points.
- **Search visibility:** lower March GSC impressions receives more points.

The total score is the sum of the two signal scores. Higher scores receive higher refresh priority.

Reason codes:
- `STALE_AND_LOW_VISIBILITY` — both signals indicate priority
- `STALE` — staleness indicates priority
- `LOW_VISIBILITY` — low visibility indicates priority
- `REVIEW` — neither signal crosses the priority threshold

Action labels:
- `REFRESH` for items with at least one priority signal
- `REVIEW` otherwise

This is a decision-support baseline, not a claim that the selected content will improve after refresh.

In [27]:
# Build the baseline ranked queue

queue = con.execute(f"""
WITH content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        DATE_DIFF(
            'day',
            content_updated_date,
            DATE '2026-03-31'
        ) AS days_stale
    FROM {DIM_CONTENT}
    WHERE content_updated_date IS NOT NULL
      AND is_deleted IS NOT TRUE
),
performance AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks
    FROM {PERF_MARCH}
    GROUP BY 1, 2
),
scored AS (
    SELECT
        c.client_hash_id,
        c.content_hash_id,
        c.content_updated_date,
        c.days_stale,
        COALESCE(p.march_impressions, 0) AS march_impressions,
        COALESCE(p.march_clicks, 0) AS march_clicks,

        CASE
            WHEN c.days_stale >= 365 THEN 3
            WHEN c.days_stale >= 180 THEN 2
            WHEN c.days_stale >= 90 THEN 1
            ELSE 0
        END AS staleness_score,

        CASE
            WHEN COALESCE(p.march_impressions, 0) < 100 THEN 3
            WHEN COALESCE(p.march_impressions, 0) < 1000 THEN 2
            WHEN COALESCE(p.march_impressions, 0) < 10000 THEN 1
            ELSE 0
        END AS visibility_score
    FROM content c
    LEFT JOIN performance p
      USING (client_hash_id, content_hash_id)
),
final AS (
    SELECT
        *,
        staleness_score + visibility_score AS score,

        CASE
            WHEN staleness_score > 0 AND visibility_score > 0
                THEN 'STALE_AND_LOW_VISIBILITY'
            WHEN staleness_score > 0
                THEN 'STALE'
            WHEN visibility_score > 0
                THEN 'LOW_VISIBILITY'
            ELSE 'REVIEW'
        END AS reason_code,

        CASE
            WHEN staleness_score > 0 OR visibility_score > 0
                THEN 'REFRESH'
            ELSE 'REVIEW'
        END AS action
    FROM scored
)
SELECT
    *,
    ROW_NUMBER() OVER (
        ORDER BY score DESC, march_impressions ASC, content_hash_id
    ) AS rank
FROM final
ORDER BY rank
""").df()

# Save the ranked queue
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(f"Rows in queue: {len(queue):,}")
print(f"Top score: {queue['score'].max()}")
print("\nTop 10:")
display(queue.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in queue: 418,047
Top score: 5

Top 10:


,client_hash_id,content_hash_id,content_updated_date,days_stale,march_impressions,march_clicks,staleness_score,visibility_score,score,reason_code,action,rank
0,client_65de48885f4ef01b,content_000308203900dd8b,2025-06-01,303,0.0,0.0,2,3,5,STALE_AND_LOW_VISIBILITY,REFRESH,1
1,client_2b4306c3ed003f01,content_000a6e4e27a0f725,2025-09-21,191,0.0,0.0,2,3,5,STALE_AND_LOW_VISIBILITY,REFRESH,2
2,client_625b6439094e23e4,content_000e7d41ec22c990,2025-09-21,191,0.0,0.0,2,3,5,STALE_AND_LOW_VISIBILITY,REFRESH,3
3,client_625b6439094e23e4,content_0058408e6a672456,2025-09-17,195,0.0,0.0,2,3,5,STALE_AND_LOW_VISIBILITY,REFRESH,4
4,client_65de48885f4ef01b,content_0080de4e5412b7e5,2025-06-01,303,0.0,0.0,2,3,5,STALE_AND_LOW_VISIBILITY,REFRESH,5
5,client_b10cb2997d0c7c86,content_008c3371c540e4cb,2025-08-09,234,0.0,0.0,2,3,5,STALE_AND_LOW_VISIBILITY,REFRESH,6
6,client_2b4306c3ed003f01,content_008df8ff3e17afd0,2025-09-21,191,0.0,0.0,2,3,5,STALE_AND_LOW_VISIBILITY,REFRESH,7
7,client_2b4306c3ed003f01,content_00ce9b2a43026011,2025-09-21,191,0.0,0.0,2,3,5,STALE_AND_LOW_VISIBILITY,REFRESH,8
8,client_625b6439094e23e4,content_00fdda8b55acf000,2025-09-17,195,0.0,0.0,2,3,5,STALE_AND_LOW_VISIBILITY,REFRESH,9
9,client_65de48885f4ef01b,content_0139917e90d8aae6,2025-06-01,303,0.0,0.0,2,3,5,STALE_AND_LOW_VISIBILITY,REFRESH,10


## 3. Top-20 review

I reviewed the 20 highest-ranked items from the baseline queue.

The action is `REFRESH` when the baseline identifies at least one priority signal. The confidence note describes how directly the observed signals support the action. The final column records what could make the recommendation wrong, so the queue is treated as decision-support rather than certainty.

In [28]:
# Review the top 20 ranked items

top20_review = queue.head(20).copy()

top20_review["confidence_note"] = (
    "Both baseline signals are present, but zero observed search activity "
    "can also reflect low demand or other causes."
)

top20_review["what_would_make_wrong"] = (
    "The content may have an intentional low-visibility role, "
    "recent changes may not yet be reflected, or the observed March "
    "signals may not indicate that a refresh would help."
)

top20_review = top20_review[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_wrong",
        "days_stale",
        "march_impressions",
        "march_clicks",
        "score",
    ]
]

display(top20_review)

,rank,content_hash_id,action,reason_code,confidence_note,what_would_make_wrong,days_stale,march_impressions,march_clicks,score
0,1,content_000308203900dd8b,REFRESH,STALE_AND_LOW_VISIBILITY,"Both baseline signals are present, but zero ob...",The content may have an intentional low-visibi...,303,0.0,0.0,5
1,2,content_000a6e4e27a0f725,REFRESH,STALE_AND_LOW_VISIBILITY,"Both baseline signals are present, but zero ob...",The content may have an intentional low-visibi...,191,0.0,0.0,5
2,3,content_000e7d41ec22c990,REFRESH,STALE_AND_LOW_VISIBILITY,"Both baseline signals are present, but zero ob...",The content may have an intentional low-visibi...,191,0.0,0.0,5
3,4,content_0058408e6a672456,REFRESH,STALE_AND_LOW_VISIBILITY,"Both baseline signals are present, but zero ob...",The content may have an intentional low-visibi...,195,0.0,0.0,5
4,5,content_0080de4e5412b7e5,REFRESH,STALE_AND_LOW_VISIBILITY,"Both baseline signals are present, but zero ob...",The content may have an intentional low-visibi...,303,0.0,0.0,5
5,6,content_008c3371c540e4cb,REFRESH,STALE_AND_LOW_VISIBILITY,"Both baseline signals are present, but zero ob...",The content may have an intentional low-visibi...,234,0.0,0.0,5
6,7,content_008df8ff3e17afd0,REFRESH,STALE_AND_LOW_VISIBILITY,"Both baseline signals are present, but zero ob...",The content may have an intentional low-visibi...,191,0.0,0.0,5
7,8,content_00ce9b2a43026011,REFRESH,STALE_AND_LOW_VISIBILITY,"Both baseline signals are present, but zero ob...",The content may have an intentional low-visibi...,191,0.0,0.0,5
8,9,content_00fdda8b55acf000,REFRESH,STALE_AND_LOW_VISIBILITY,"Both baseline signals are present, but zero ob...",The content may have an intentional low-visibi...,195,0.0,0.0,5
9,10,content_0139917e90d8aae6,REFRESH,STALE_AND_LOW_VISIBILITY,"Both baseline signals are present, but zero ob...",The content may have an intentional low-visibi...,303,0.0,0.0,5


In [29]:
import json

metrics = {
    "rows_in_queue": int(len(queue)),
    "top_score": int(queue["score"].max()),
    "top20_refresh_count": int((top20_review["action"] == "REFRESH").sum()),
    "top20_stale_and_low_visibility_count": int(
        (top20_review["reason_code"] == "STALE_AND_LOW_VISIBILITY").sum()
    )
}

with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(metrics)

{'rows_in_queue': 418047, 'top_score': 5, 'top20_refresh_count': 20, 'top20_stale_and_low_visibility_count': 20}


## 4. Weak picks + leakage check

The top-ranked picks are directionally supported by both observed signals, but they are not guaranteed refresh opportunities. A content item with zero March impressions and clicks may be intentionally low-demand, newly targeted, technically unavailable, or otherwise unsuitable for refresh.

The baseline uses only information available at the March 31, 2026 decision point:
- content update date
- March search impressions
- March search clicks

It does not use future-window outcomes, `trend_direction`, `trend_pct`, or product-decision flags.

In [30]:
# Section 4: inspect weak picks and verify the baseline inputs

print("Top-20 reason-code counts:")
display(
    top20_review["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="n")
)

print("\nTop-20 action counts:")
display(
    top20_review["action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="n")
)

print("\nLeakage check:")
print("Future-window fields used: none")
print("Label-derived fields used: none")
print("Product-decision flags used: none")

# Confirm the queue contains only the intended baseline inputs.
baseline_inputs = {
    "content_updated_date",
    "days_stale",
    "march_impressions",
    "march_clicks",
    "staleness_score",
    "visibility_score",
    "score",
    "reason_code",
    "action",
    "rank",
    "client_hash_id",
    "content_hash_id",
}

unexpected = set(queue.columns) - baseline_inputs

print(f"Unexpected queue columns: {sorted(unexpected)}")

Top-20 reason-code counts:


,reason_code,n
0,STALE_AND_LOW_VISIBILITY,20



Top-20 action counts:


,action,n
0,REFRESH,20



Leakage check:
Future-window fields used: none
Label-derived fields used: none
Product-decision flags used: none
Unexpected queue columns: []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.